# Retrain v4 multi-seed: training-stochastic CIs for the v3-recipe post-v0.34 fine-tune

**Hypothesis under test (2026-04-28).** The single-seed v4 retrain (seed=42) showed macro NDCG@10 lift +0.0204 over base BGE-small with 95% CI [+0.0154, +0.0256] (paired bootstrap, eval-stochastic). The CI excludes zero but the magnitude is DIRECTIONAL per the pre-committed threshold of 0.04 absolute. This notebook adds **training-stochastic** CIs by retraining v4 with multiple seeds and reporting the cross-seed range.

**Configuration.** Same recipe as v4 (arm_vol from H-R9 ablation: temperature=0.5, total_triples=60000, MNRL, 2 epochs at lr=3e-6, batch=64). Seeds: {0, 1, 42, 100} -- four runs, each saving its trained model and per-query NDCG sidecar.

**Wall time on Colab T4.** ~12 min per seed for training + ~15 min per seed for BEIR eval (CUDA, 5 datasets). 4 seeds = ~110 min total.

**Outputs.**
- `/content/v4_seed_S/` -- trained model per seed
- `/content/perquery_seed_S.json` -- per-query NDCG sidecar per seed (for paired bootstrap on Mac)
- `/content/multiseed_summary.json` -- aggregate macro per seed
- `/content/v4_multiseed_artifacts.tar.gz` -- bundled download

After running, on Mac compute cross-seed CI on the macro deltas via `experiments/multiseed_aggregate.py` (or inline analysis).

In [ ]:
# Cell 1: Setup -- clone the experiment branch (has per-query sidecar patch + bootstrap script).
!pip install -q 'sentence-transformers>=3' torch 'accelerate>=1.1.0'
!rm -rf /content/vstash
!git clone --branch experiment/retrain-v4-post-v034-validation https://github.com/stffns/vstash.git /content/vstash
%cd /content/vstash
!pip install -q -e .
!python -c "import vstash; print('vstash', vstash.__version__)"
!grep -n 'per_query' experiments/beir_benchmark.py | head -3

In [ ]:
# Cell 2: Download BEIR + ingest each training dataset once (reused across seeds).
import os
import sys

os.chdir("/content/vstash")
sys.path.insert(0, "/content/vstash")
os.makedirs("experiments/data", exist_ok=True)

from experiments.beir_benchmark import download_beir, load_beir
from sentence_transformers import SentenceTransformer
from vstash.store import VstashStore

BASE_MODEL = "BAAI/bge-small-en-v1.5"
TRAIN_DATASETS = ["scifact", "nfcorpus", "fiqa"]  # v3/v4 training set
encoder = SentenceTransformer(BASE_MODEL, device="cuda")
stores = {}
per_dataset = {}

for name in TRAIN_DATASETS:
    cache = download_beir(name)
    corpus, queries, qrels = load_beir(cache)
    print(f"[{name}] corpus={len(corpus)} queries={len(queries)} qrels={len(qrels)}")
    db_path = f"/content/store_{name}.db"
    if os.path.exists(db_path):
        os.remove(db_path)
    store = VstashStore(db_path, embedding_dim=384)
    doc_ids = list(corpus.keys())
    BATCH = 256
    for i in range(0, len(doc_ids), BATCH):
        batch_ids = doc_ids[i : i + BATCH]
        texts = [
            (corpus[d].get("title", "") + "\n" + corpus[d].get("text", "")).strip()
            for d in batch_ids
        ]
        embs = encoder.encode(texts, normalize_embeddings=True, show_progress_bar=False)
        store.add_documents_batch(
            [
                {
                    "path": f"{name}://{doc_id}",
                    "title": corpus[doc_id].get("title", ""),
                    "chunks": [text],
                    "embeddings": [emb.tolist()],
                    "source_type": "text",
                }
                for doc_id, text, emb in zip(batch_ids, texts, embs)
            ]
        )
    stores[name] = store
    per_dataset[name] = {"queries": queries, "qrels": qrels}
    print(f"  {name}: ingested {store.stats().chunks} chunks")

In [ ]:
# Cell 3: Build per-dataset eval queries from real qrels (T1.5 v5 recipe).
from vstash.retrain import qrels_to_eval_queries

eval_queries_by_dataset = {}
for name, bundle in per_dataset.items():
    eqs = qrels_to_eval_queries(
        queries=bundle["queries"],
        qrels=bundle["qrels"],
        path_for_doc_id=lambda doc_id, d=name: f"{d}://{doc_id}",
    )
    eval_queries_by_dataset[name] = eqs
    print(f"[{name}] eval_queries: {len(eqs)}")

In [ ]:
# Cell 4: Multi-seed retrain + eval loop.
# Each seed produces a model + per-query sidecar JSON.
import time
import json
import shutil
import subprocess
from vstash.retrain import retrain_multi

EVAL_NOISE = max(max(s.stats().chunks for s in stores.values()), 10000)
SEEDS = [0, 1, 42, 100]
results_per_seed = {}

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

for seed in SEEDS:
    print(f"\n{'=' * 70}\n  SEED {seed}\n{'=' * 70}")
    output_path = f"/content/v4_seed_{seed}"
    for suffix in ("", ".candidate", ".old"):
        p = output_path + suffix
        if os.path.exists(p):
            shutil.rmtree(p)

    t0 = time.time()
    result = retrain_multi(
        stores=stores,
        base_model=BASE_MODEL,
        output_path=output_path,
        sampling="temperature",
        temperature=0.5,
        total_triples=60000,
        epochs=2,
        lr=3e-6,
        batch_size=64,
        eval_noise_size=EVAL_NOISE,
        bulk_mine=True,
        bulk_mine_device="cuda",
        seed=seed,
        eval_queries_by_dataset=eval_queries_by_dataset,
        min_gain=-1.0,
    )
    train_min = (time.time() - t0) / 60
    print(f"  seed={seed} training: {train_min:.1f} min")

    # BEIR eval all 5 datasets via patched beir_benchmark.py
    t0 = time.time()
    proc = subprocess.run(
        [
            "python",
            "-m",
            "experiments.beir_benchmark",
            "--no-chroma",
            "--device",
            "cuda",
            "--model",
            output_path,
        ],
        cwd="/content/vstash",
        capture_output=True,
        text=True,
    )
    eval_min = (time.time() - t0) / 60
    print(f"  seed={seed} BEIR eval: {eval_min:.1f} min, returncode={proc.returncode}")
    if proc.returncode != 0:
        print("STDERR tail:", proc.stderr[-500:])

    # Locate the per-query sidecar (path-slug based filename) and rename to seed-tagged.
    pq_dir = "/content/vstash/experiments/results"
    candidates = [
        f for f in os.listdir(pq_dir) if f.startswith("beir_perquery_") and f.endswith(".json")
    ]
    candidates.sort(key=lambda f: os.path.getmtime(os.path.join(pq_dir, f)), reverse=True)
    if candidates:
        src = os.path.join(pq_dir, candidates[0])
        dst = f"/content/perquery_seed_{seed}.json"
        shutil.copy(src, dst)
        # Also save the aggregate.
        shutil.copy(
            "/content/vstash/experiments/results/beir_benchmark.json",
            f"/content/aggregate_seed_{seed}.json",
        )
        with open(f"/content/aggregate_seed_{seed}.json") as f:
            agg = json.load(f)
        per_ds = {r["dataset"]: r["vstash"]["ndcg_10"] for r in agg["results"]}
        per_ds["_macro"] = sum(per_ds.values()) / len(per_ds)
        results_per_seed[seed] = per_ds
        print(f"  per-dataset NDCG@10 (seed {seed}):")
        for ds in sorted(per_ds):
            print(f"    {ds:<10} {per_ds[ds]:.4f}")

with open("/content/multiseed_summary.json", "w") as f:
    json.dump(results_per_seed, f, indent=2)
print("\n  Multiseed summary saved.")

In [ ]:
# Cell 5: Cross-seed summary table -- macro and per-dataset spread.
import statistics

datasets = sorted({d for seed in results_per_seed for d in results_per_seed[seed] if d != "_macro"})
datasets += ["_macro"]

print(f"{'dataset':<10} " + " ".join(f"seed={s:<6}" for s in SEEDS) + " mean    std    range")
print("-" * 80)
for ds in datasets:
    vals = [
        results_per_seed[s][ds]
        for s in SEEDS
        if s in results_per_seed and ds in results_per_seed[s]
    ]
    if not vals:
        continue
    mean = statistics.mean(vals)
    stdev = statistics.stdev(vals) if len(vals) > 1 else 0.0
    rng = max(vals) - min(vals)
    row = f"{ds:<10} " + " ".join(f"{v:.4f}    " for v in vals)
    row += f"{mean:.4f}  {stdev:.4f}  {rng:.4f}"
    print(row)

# v3 reference (single-seed=42, pre-v0.34 trained, Mac CPU measurement)
v3_ref = {
    "scifact": 0.7705,
    "nfcorpus": 0.3755,
    "fiqa": 0.4648,
    "scidocs": 0.1954,
    "arguana": 0.4318,
}
v3_ref["_macro"] = sum(v3_ref.values()) / len(v3_ref)
print("\n  v3 reference (single seed, pre-v0.34 trained):")
for ds in datasets:
    if ds in v3_ref:
        print(f"    {ds:<10} {v3_ref[ds]:.4f}")

# Read of result: if cross-seed std on v4 macro is small (~0.001-0.003) and
# v3 macro falls within mean +/- 2*std, training-stochastic noise explains
# the v4-vs-v3 difference. If v3 falls outside the band, the training-side
# effect of the cosine fix is real beyond seed noise.

In [ ]:
# Cell 6: Bundle artifacts and download.
import os
from google.colab import files

!cd /content && tar czf v4_multiseed_artifacts.tar.gz \
    perquery_seed_*.json \
    aggregate_seed_*.json \
    multiseed_summary.json

# Bundle each seed's model separately (model.safetensors is ~130MB each, ~530MB total)
for seed in SEEDS:
    if os.path.exists(f"/content/v4_seed_{seed}"):
        !cd /content && tar czf v4_seed_{seed}_model.tar.gz v4_seed_{seed}

files.download("/content/v4_multiseed_artifacts.tar.gz")
for seed in SEEDS:
    p = f"/content/v4_seed_{seed}_model.tar.gz"
    if os.path.exists(p):
        files.download(p)